# 03 — Sigma-only Adam fitting

The production fitter holds amplitudes and centroids fixed at their initialised values and optimises only the spreads (sigmas) with Adam, minimising the per-pixel mean-squared error. This notebook reproduces the exact loss and the Adam update from `SigmaScan.per_pixel_loss` and `SigmaScan.adam_scan` (`pipelines.param_pipeline.sigma`) in NumPy, then checks the fitted sigma against a known true sigma.

Modules exercised (reproduced on CPU):

- `pipelines.param_pipeline.sigma.SigmaScan` (loss and Adam scan reproduced inline; the production kernel is JAX-jitted and JAX is not installed here)

Synthetic profile, fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


## Reproduce the loss and Adam scan

`per_pixel_loss` evaluates a fixed-amplitude, fixed-mu mixture at the current sigmas and returns the MSE against the target profile. `adam_scan` runs bias-corrected Adam with the same hyper-parameters and clips sigma into `[sigma_lower, sigma_upper]` at every step, exactly as in the source.

In [ ]:
def per_pixel_pred(sigmas, height_axis, amps, mus):
    safe_s2 = 2.0 * np.maximum(sigmas, 1e-6) ** 2
    diff    = height_axis[None, :] - mus[:, None]
    expon   = np.clip(-(diff ** 2) / safe_s2[:, None], -100.0, 0.0)
    return (amps[:, None] * np.exp(expon)).sum(axis=0)

def per_pixel_loss(sigmas, height_axis, profile, amps, mus):
    pred = per_pixel_pred(sigmas, height_axis, amps, mus)
    return float(np.mean((pred - profile) ** 2))

def loss_grad(sigmas, height_axis, profile, amps, mus, eps=1e-5):
    g = np.zeros_like(sigmas)
    for k in range(sigmas.size):
        sp = sigmas.copy(); sp[k] += eps
        sm = sigmas.copy(); sm[k] -= eps
        g[k] = (per_pixel_loss(sp, height_axis, profile, amps, mus) - per_pixel_loss(sm, height_axis, profile, amps, mus)) / (2 * eps)
    return g

def adam_scan(sigmas_init, height_axis, profile, amps, mus, sigma_lower, sigma_upper, n_steps=2000, lr=0.2, b1=0.95, b2=0.999):
    eps = 1e-8
    s   = np.clip(sigmas_init.astype(np.float64), sigma_lower, sigma_upper)
    m   = np.zeros_like(s)
    v   = np.zeros_like(s)
    history = []
    for t in range(n_steps):
        g  = loss_grad(s, height_axis, profile, amps, mus)
        m  = b1 * m + (1.0 - b1) * g
        v  = b2 * v + (1.0 - b2) * g * g
        tf = t + 1.0
        s  = s - lr * (m / (1.0 - b1 ** tf)) / (np.sqrt(v / (1.0 - b2 ** tf)) + eps)
        s  = np.clip(s, sigma_lower, sigma_upper)
        history.append(per_pixel_loss(s, height_axis, profile, amps, mus))
    return s, np.array(history)

print('loss and adam scan reproduced')

## Single-component fit with known sigma

Amplitude and centroid are fixed at their true values; only sigma is fitted. The bounds mirror the production setup: `sigma_lower = dh` (one height step) and `sigma_upper = h_span / 2`.

In [ ]:
H            = 200
height_axis  = np.linspace(0.0, 40.0, H).astype(np.float64)
dh           = float(height_axis[1] - height_axis[0])
h_span       = float(height_axis[-1] - height_axis[0])
sigma_lower  = dh
sigma_upper  = h_span / 2.0

true_amp, true_mu, true_sigma = 1.0, 18.0, 3.0
profile = gaussian_mixture_profile(height_axis, [(true_amp, true_mu, true_sigma)], noise_std=0.01, rng=rng).astype(np.float64)

amps = np.array([true_amp])
mus  = np.array([true_mu])
s0   = np.array([0.5 * true_sigma])

s_fit, hist = adam_scan(s0, height_axis, profile, amps, mus, sigma_lower, sigma_upper, n_steps=400)
print(f'true sigma   : {true_sigma:.4f}')
print(f'init sigma   : {float(s0[0]):.4f}')
print(f'fitted sigma : {float(s_fit[0]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(height_axis, profile, color='black', lw=1.4, label='target profile')
ax.plot(height_axis, per_pixel_pred(s0,    height_axis, amps, mus), color='tab:orange', lw=1.2, ls=':',  label='initial sigma')
ax.plot(height_axis, per_pixel_pred(s_fit, height_axis, amps, mus), color='tab:red',    lw=1.3, ls='--', label='fitted sigma')
ax.set_xlabel('height h [m]')
ax.set_ylabel('intensity')
ax.set_title('Sigma fit vs target')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(np.arange(hist.size), hist, color='tab:blue', lw=1.2)
ax.set_yscale('log')
ax.set_xlabel('Adam step')
ax.set_ylabel('MSE loss')
ax.set_title('Convergence of sigma-only Adam')
fig.tight_layout()
plt.show()

## Expected outcome

The fitted-sigma curve should overlay the target profile while the initial-sigma curve is visibly too narrow, and the fitted sigma should match the true value (3.0 m) to within the noise level. The loss curve should decrease monotonically and flatten, confirming the reproduced Adam scan converges to the correct spread.